# RetinaVLM: 논문 결과 재현 (Inference Only)

이 노트북은 **학습(Fine-tuning) 없이** 이미 학습된 `RetinaVLM-Specialist` 가중치를 사용하여 OCT 이미지를 분석합니다.

### 모델 구성
- **Vision Encoder**: ResNet50 (Pre-trained on Specialist task)
- **Projection Layer**: Pre-trained & Aligned Linear Layer
- **LLM**: Llama3-8B-Instruct

이 코드는 `dataset/processed_images` 경로의 실제 데이터를 사용하여 논문의 성능을 확인합니다.

In [ ]:
import os, sys, torch, glob
from PIL import Image
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from hydra import initialize, compose

# 프로젝트 경로 설정 (현재 폴더를 루트로 설정)
PROJECT_DIR = os.path.abspath(".")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from models.retinavlm_wrapper import load_retinavlm_specialist_from_hf

# 장치 설정
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# =====================================================
#  학습된 RetinaVLM-Specialist 모델 로드
# =====================================================

# 경로 중복 방지를 위해 PROJECT_DIR 사용
with initialize(version_base=None, config_path="configs"):
    config = compose(config_name="default")

print("Loading pre-trained RetinaVLM-Specialist from HuggingFace...")
model = load_retinavlm_specialist_from_hf(config)
model.to(DEVICE)
model.eval()
print("Model loaded successfully!")

In [ ]:
# =====================================================
#  데이터 로드 및 시각화
# =====================================================

# 경로에서 SpecialistVLMs 중복 제거
IMAGE_DIR = os.path.join(PROJECT_DIR, "dataset", "processed_images")
image_paths = sorted([p for p in glob.glob(os.path.join(IMAGE_DIR, "*.png")) if "README" not in p])
print(f"Found {len(image_paths)} images at {IMAGE_DIR}")

def show_samples(paths, n=5):
    if not paths:
        print("No images found! Check IMAGE_DIR path.")
        return
    n = min(n, len(paths))
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    if n == 1: axes = [axes]
    for i in range(n):
        img = Image.open(paths[i])
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"Sample #{i}")
        axes[i].axis('off')
    plt.show()

show_samples(image_paths)

In [ ]:
# =====================================================
#  추론(Inference) 실행
# =====================================================

def run_inference(image_index, query):
    if not image_paths:
        print("Error: No images loaded.")
        return
        
    image_path = image_paths[image_index]
    img = Image.open(image_path).convert('L')
    
    print(f"Analyzing: {os.path.basename(image_path)}")
    print(f"Query: {query}")
    
    with torch.no_grad():
        response = model.forward([img], [query], max_new_tokens=300)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.show()
    
    print("\n[AI Analysis Result]")
    print("=" * 60)
    print(response[0])
    print("=" * 60)

# 테스트 실행 (첫 번째 이미지)
test_query = "Write a detailed clinical report describing this OCT scan. Identify biomarkers such as drusen, fluid, or atrophy."
run_inference(0, test_query)